In [1]:
import h5py
import numpy as np

In [2]:
reference_mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\reference_packet.mat'

In [3]:
with h5py.File(reference_mat_file, 'r') as f:
    reference_packet = np.array(f['packets'])  # shape: (recordLength,)
    # Access metadata
    metadata = f['metadata']
    sample_rate = metadata['sample_rate'][0][0]
    trigger_level = metadata['trigger_level'][0][0]
    record_length = metadata['record_length'][0][0]
    num_frames = metadata['num_frames'][0][0]

In [4]:
reference_packet = reference_packet[0, :]

In [5]:
reference_packet

array([ 0.03296875,  0.0140625 , -0.00929687, ..., -0.01664062,
       -0.02210938,  0.00671875], shape=(250000,))

In [6]:
def decode_ethernet(samples, fs=1.25e9, verbose=True):
    """
    Decode a 10BASE-T Ethernet packet from a voltage signal.

    Parameters
    ----------
    samples : np.ndarray
        Array of voltage samples from the oscilloscope.
    fs : float
        Sampling frequency in Hz (default 1.25e9).
    verbose : bool
        If True, print progress and diagnostic information.

    Returns
    -------
    bytes
        The Ethernet payload (data after the header).
    """
    # Constants for 10BASE-T
    BIT_RATE = 10e6                 # 10 Mbps
    BIT_PERIOD_NS = 1e3 / BIT_RATE  # 100 ns
    HALF_BIT_NS = BIT_PERIOD_NS / 2 # 50 ns
    SAMPLE_PERIOD_NS = 1e9 / fs     # 0.8 ns
    SAMPLES_PER_HALF_BIT = HALF_BIT_NS / SAMPLE_PERIOD_NS  # 62.5

    # Remove DC offset (assume symmetric signal)
    offset = np.median(samples)
    signal = samples - offset
    if verbose:
        print(f"DC offset removed: {offset:.3f} V")

    # Find zero crossings with sub‑sample accuracy
    sign = np.sign(signal)
    # Indices where sign changes (including zero exactly)
    cross_idx = []
    for i in range(len(signal)-1):
        if sign[i] * sign[i+1] < 0:  # opposite signs
            # Linear interpolation for fractional index
            t = i + signal[i] / (signal[i] - signal[i+1])
            cross_idx.append(t)
        elif signal[i] == 0:          # exactly zero (rare)
            cross_idx.append(i)
    cross_idx = np.array(cross_idx)

    if len(cross_idx) < 2:
        raise ValueError("Not enough zero crossings – signal may be idle or noisy")

    # Convert crossing times to nanoseconds
    cross_times_ns = cross_idx * SAMPLE_PERIOD_NS

    # Compute intervals between successive crossings
    intervals_ns = np.diff(cross_times_ns)

    # Classify each interval as 1 or 2 half‑bit periods
    half_units = np.round(intervals_ns / HALF_BIT_NS).astype(int)
    # Sanity check: most intervals should be 1 or 2
    unique, counts = np.unique(half_units, return_counts=True)
    if verbose:
        print("Interval half‑unit counts:", dict(zip(unique, counts)))
    if not all(u in [1,2] for u in unique):
        print("Warning: some intervals are not 1 or 2 half‑bits – check signal quality")

    # Determine direction of each crossing (rising or falling)
    # Use the sample immediately after the crossing (round to nearest integer)
    directions = []
    for idx in cross_idx:
        i = int(round(idx))
        if i >= len(signal)-1:
            i = len(signal)-2
        if signal[i+1] > signal[i]:
            directions.append('rising')
        else:
            directions.append('falling')
    directions = np.array(directions)

    # ---- Reconstruct half‑bit levels ----
    # Start with the level immediately after the first crossing
    if directions[0] == 'rising':
        current_level = 1      # positive voltage
        first_bit = 1          # rising => bit 1 (low->high)
    else:
        current_level = -1     # negative voltage
        first_bit = 0

    # Build list of levels for each half‑bit after the first crossing
    levels = []                # levels for each half‑bit interval
    for n in half_units:
        levels.extend([current_level] * n)
        current_level *= -1    # crossing flips the level

    # Prepend the missing first half of the first bit
    # For bit = 1: first half low (-1), second half high (+1)
    # For bit = 0: first half high (+1), second half low (-1)
    first_half = -1 if first_bit == 1 else 1
    full_levels = [first_half] + levels

    # Group into bits (two consecutive half‑bits)
    bits = []
    for i in range(0, len(full_levels)-1, 2):
        first = full_levels[i]
        second = full_levels[i+1]
        if first == -1 and second == 1:
            bits.append(1)
        elif first == 1 and second == -1:
            bits.append(0)
        else:
            # Invalid Manchester – maybe polarity is reversed?
            # Try swapping interpretation
            if first == 1 and second == -1:
                bits.append(1)   # actually this would be the other mapping
            else:
                bits.append(None)
                if verbose:
                    print(f"Invalid Manchester at bit {i//2}")

    # Remove possible None at the end if odd number of half‑bits
    bits = [b for b in bits if b is not None]

    if verbose:
        print(f"Recovered {len(bits)} bits")

    # ---- Locate SFD ----
    # Preamble: 7 bytes of 0x55 (LSB first) => 56 bits of 10101010...
    # SFD: 0xD5 (LSB first) => 10101011
    # So we look for the pattern 10101011
    sfd_pattern = [1,0,1,0,1,0,1,1]
    # Convert bits to list for searching
    bit_list = bits
    sfd_start = None
    for i in range(len(bit_list) - 8):
        if bit_list[i:i+8] == sfd_pattern:
            sfd_start = i
            break

    if sfd_start is None:
        # Try inverted polarity (if probe was swapped)
        inv_pattern = [0,1,0,1,0,1,0,0]  # inverted SFD
        for i in range(len(bit_list) - 8):
            if bit_list[i:i+8] == inv_pattern:
                sfd_start = i
                # Invert all bits for further processing
                bits = [1-b for b in bits]
                if verbose:
                    print("Polarity inverted – corrected.")
                break
    if sfd_start is None:
        raise ValueError("SFD not found – cannot locate frame start")

    if verbose:
        print(f"SFD found at bit index {sfd_start}")

    # Frame starts immediately after SFD (next bit)
    frame_bits = bits[sfd_start+8:]

    # Convert bits to bytes (LSB first)
    frame_bytes = []
    for i in range(0, len(frame_bits) - 7, 8):
        byte_bits = frame_bits[i:i+8]
        byte = 0
        for j, b in enumerate(byte_bits):
            if b:
                byte |= (1 << j)   # LSB first
        frame_bytes.append(byte)

    if verbose:
        print(f"Decoded {len(frame_bytes)} bytes of Ethernet frame")

    # ---- Extract payload ----
    # Ethernet II frame: Dest MAC (6), Src MAC (6), Ethertype (2), Payload, FCS (4)
    if len(frame_bytes) < 14:
        raise ValueError("Frame too short – missing header")

    dest_mac = frame_bytes[0:6]
    src_mac = frame_bytes[6:12]
    ethertype = (frame_bytes[12] << 8) | frame_bytes[13]
    payload_start = 14
    # Payload ends 4 bytes before the end (FCS)
    payload_end = len(frame_bytes) - 4
    if payload_end <= payload_start:
        print("Warning: no payload or frame too short")
        payload = b''
    else:
        payload = bytes(frame_bytes[payload_start:payload_end])

    if verbose:
        print(f"Destination MAC: {':'.join(f'{b:02x}' for b in dest_mac)}")
        print(f"Source MAC: {':'.join(f'{b:02x}' for b in src_mac)}")
        print(f"Ethertype: 0x{ethertype:04x}")
        print(f"Payload length: {len(payload)} bytes")

    return payload

# ---------------------------------------------------------------------
# Example usage (assuming the array is stored in a variable `samples`):
samples = reference_packet  # Replace with actual samples from oscilloscope
payload = decode_ethernet(samples)
print("Payload:", payload)

DC offset removed: 0.013 V
Interval half‑unit counts: {np.int64(65): np.int64(1), np.int64(80): np.int64(1), np.int64(84): np.int64(1), np.int64(85): np.int64(1), np.int64(99): np.int64(1), np.int64(107): np.int64(1), np.int64(110): np.int64(1), np.int64(111): np.int64(1), np.int64(117): np.int64(1), np.int64(118): np.int64(1), np.int64(126): np.int64(2), np.int64(132): np.int64(1), np.int64(134): np.int64(1), np.int64(135): np.int64(1), np.int64(137): np.int64(1), np.int64(143): np.int64(1), np.int64(156): np.int64(1), np.int64(160): np.int64(1), np.int64(161): np.int64(1), np.int64(162): np.int64(1), np.int64(164): np.int64(1), np.int64(166): np.int64(1), np.int64(174): np.int64(1), np.int64(181): np.int64(2), np.int64(186): np.int64(1), np.int64(187): np.int64(1), np.int64(188): np.int64(1), np.int64(189): np.int64(2), np.int64(191): np.int64(1), np.int64(204): np.int64(1), np.int64(205): np.int64(1), np.int64(207): np.int64(1), np.int64(210): np.int64(1), np.int64(218): np.int64(1)

KeyboardInterrupt: 

In [14]:
"""
10BASE-T Ethernet Packet Decoder
Decodes raw oscilloscope samples (1.25 GSa/s) into Ethernet frames.

Input:  NumPy array of voltage samples, shape=(250000,)
        → at 1.25 GSa/s = 200 µs of capture = 2000 bit periods @ 10 Mbps
"""

import numpy as np
from collections import namedtuple

# ── Constants ────────────────────────────────────────────────────────────────
SAMPLE_RATE      = 1.25e9          # 1.25 GSa/s
BIT_RATE         = 10e6            # 10 Mbps
SAMPLES_PER_BIT  = int(SAMPLE_RATE / BIT_RATE)   # 125
HALF_BIT         = SAMPLES_PER_BIT // 2           # 62

EtherType = namedtuple("EtherType", ["value", "name"])
ETHERTYPE_MAP = {
    0x0800: "IPv4",
    0x0806: "ARP",
    0x86DD: "IPv6",
    0x8100: "VLAN (802.1Q)",
    0x8847: "MPLS",
    0x88CC: "LLDP",
}

# ── Step 1: Preprocessing ────────────────────────────────────────────────────

def remove_dc_offset(samples: np.ndarray) -> np.ndarray:
    """Remove DC bias — scope ground reference may drift."""
    return samples - np.mean(samples)


def apply_lowpass(samples: np.ndarray, cutoff_ratio: float = 0.02) -> np.ndarray:
    """
    Simple moving-average low-pass filter to suppress HF noise.
    cutoff_ratio: fraction of sample rate (0.02 → ~25 MHz cutoff)
    """
    window = max(1, int(1.0 / cutoff_ratio))
    kernel = np.ones(window) / window
    return np.convolve(samples, kernel, mode='same')


# ── Step 2: Threshold & Edge Detection ───────────────────────────────────────

def detect_threshold(samples: np.ndarray) -> float:
    """Auto-threshold: midpoint between signal min and max (hysteresis centre)."""
    return (np.max(samples) + np.min(samples)) / 2.0


def find_edges(samples: np.ndarray, threshold: float) -> np.ndarray:
    """
    Return indices of zero-crossings (rising + falling edges).
    Uses sign-change detection on threshold-subtracted signal.
    """
    shifted   = samples - threshold
    signs     = np.sign(shifted)
    signs[signs == 0] = 1           # treat exactly-on-threshold as positive
    crossings = np.where(np.diff(signs))[0]
    return crossings


# ── Step 3: Clock Recovery via PLL ───────────────────────────────────────────

def recover_clock(edges: np.ndarray, samples_per_bit: int) -> np.ndarray:
    """
    Lightweight digital PLL: tracks expected bit-centre positions,
    snapping to nearby edges when found.

    Returns array of sample indices at each recovered bit centre.
    """
    if len(edges) == 0:
        return np.array([], dtype=int)

    # Seed the clock from the first edge
    clock_phase  = edges[0] + HALF_BIT     # first expected bit centre
    clock_period = float(samples_per_bit)
    alpha        = 0.05                    # PLL loop gain (lower = more inertia)

    bit_centres  = []
    edge_set     = set(edges.tolist())
    total        = int(edges[-1] + clock_period * 2) if len(edges) > 0 else 0

    pos = clock_phase
    while pos < total:
        bit_centres.append(int(pos))

        # Look for a nearby edge to correct phase
        search_range = int(clock_period * 0.4)
        nearby = [e for e in range(int(pos) - search_range,
                                   int(pos) + search_range)
                  if e in edge_set]
        if nearby:
            nearest_edge = min(nearby, key=lambda e: abs(e - pos))
            # Edge should fall at half-bit before centre → shift by HALF_BIT
            ideal_centre = nearest_edge + HALF_BIT
            error        = ideal_centre - pos
            pos         += clock_period + alpha * error
        else:
            pos += clock_period

    return np.array(bit_centres, dtype=int)


# ── Step 4: Manchester Decode ─────────────────────────────────────────────────

def decode_manchester(samples: np.ndarray,
                      bit_centres: np.ndarray,
                      threshold: float) -> list[int]:
    """
    Manchester decoding:
      Sample first half  → level A  (samples_per_bit//4 before centre)
      Sample second half → level B  (samples_per_bit//4 after centre)
      Transition low→high (A<thr, B>thr) = 1
      Transition high→low (A>thr, B<thr) = 0
    """
    bits    = []
    quarter = SAMPLES_PER_BIT // 4

    for centre in bit_centres:
        idx_a = centre - quarter
        idx_b = centre + quarter

        if idx_a < 0 or idx_b >= len(samples):
            continue

        a = samples[idx_a] > threshold
        b = samples[idx_b] > threshold

        if   not a and b:   bits.append(1)     # low→high = 1
        elif a and not b:   bits.append(0)     # high→low = 0
        # identical levels = invalid symbol (noise / idle) → skip
        else:               bits.append(-1)    # mark invalid

    return bits


# ── Step 5: Frame Synchronisation ────────────────────────────────────────────

PREAMBLE_BITS = [1, 0] * 28          # 56 alternating bits
SFD_BITS      = [1, 0, 1, 0, 1, 0, 1, 1]   # 10101011

def find_sfd(bits: list[int]) -> int:
    """
    Scan for Start Frame Delimiter (SFD = 10101011).
    Returns index of the first data bit after SFD, or -1 if not found.
    """
    pattern = SFD_BITS
    for i in range(len(bits) - len(pattern)):
        window = bits[i:i + len(pattern)]
        if window == pattern:
            return i + len(pattern)
    return -1


# ── Step 6: Bit stream → bytes ────────────────────────────────────────────────

def bits_to_bytes(bits: list[int]) -> bytes:
    """Convert a flat list of bits (MSB-first per byte) to a bytes object."""
    # Drop any invalid bits at the end
    clean = [b for b in bits if b in (0, 1)]
    # Pad to multiple of 8
    pad = (8 - len(clean) % 8) % 8
    clean += [0] * pad

    result = bytearray()
    for i in range(0, len(clean), 8):
        byte = 0
        for j in range(8):
            byte = (byte << 1) | clean[i + j]
        result.append(byte)
    return bytes(result)


# ── Step 7: Frame Parsing ─────────────────────────────────────────────────────

def fmt_mac(b: bytes) -> str:
    return ":".join(f"{x:02X}" for x in b)


def crc32_ethernet(data: bytes) -> int:
    """Standard CRC-32 as used by Ethernet (poly 0x04C11DB7, reflected)."""
    import struct
    crc = 0xFFFFFFFF
    for byte in data:
        crc ^= byte
        for _ in range(8):
            if crc & 1:
                crc = (crc >> 1) ^ 0xEDB88320
            else:
                crc >>= 1
    return crc ^ 0xFFFFFFFF


def parse_ethernet_frame(frame_bytes: bytes) -> dict:
    """
    Parse an Ethernet II frame.
    Minimum frame: 14 bytes header + 46 bytes payload + 4 bytes FCS = 64 bytes
    """
    result = {"raw_hex": frame_bytes.hex()}

    if len(frame_bytes) < 14:
        result["error"] = f"Too short ({len(frame_bytes)} bytes, need ≥14)"
        return result

    dst_mac    = frame_bytes[0:6]
    src_mac    = frame_bytes[6:12]
    ethertype  = int.from_bytes(frame_bytes[12:14], "big")

    result["dst_mac"]    = fmt_mac(dst_mac)
    result["src_mac"]    = fmt_mac(src_mac)
    result["ethertype"]  = f"0x{ethertype:04X}"
    result["proto_name"] = ETHERTYPE_MAP.get(ethertype, "Unknown")

    # Payload and FCS
    if len(frame_bytes) >= 18:
        payload        = frame_bytes[14:-4]
        fcs_received   = int.from_bytes(frame_bytes[-4:], "little")
        fcs_calculated = crc32_ethernet(frame_bytes[:-4])
        result["payload_len"]    = len(payload)
        result["payload_hex"]    = payload.hex()
        result["fcs_received"]   = f"0x{fcs_received:08X}"
        result["fcs_calculated"] = f"0x{fcs_calculated:08X}"
        result["fcs_valid"]      = (fcs_received == fcs_calculated)

        # Best-effort payload decode
        if ethertype == 0x0800 and len(payload) >= 20:
            result["ipv4"] = _parse_ipv4(payload)
        elif ethertype == 0x0806 and len(payload) >= 28:
            result["arp"] = _parse_arp(payload)
    else:
        result["payload_hex"] = frame_bytes[14:].hex()

    return result


def _parse_ipv4(p: bytes) -> dict:
    import socket
    ihl      = (p[0] & 0x0F) * 4
    proto    = p[9]
    src_ip   = socket.inet_ntoa(p[12:16])
    dst_ip   = socket.inet_ntoa(p[16:20])
    protos   = {1: "ICMP", 6: "TCP", 17: "UDP", 89: "OSPF"}
    return {
        "src": src_ip, "dst": dst_ip,
        "ttl": p[8],
        "proto": protos.get(proto, f"0x{proto:02X}"),
        "header_len": ihl,
        "total_len": int.from_bytes(p[2:4], "big"),
    }


def _parse_arp(p: bytes) -> dict:
    import socket
    ops = {1: "REQUEST", 2: "REPLY"}
    return {
        "operation":   ops.get(int.from_bytes(p[6:8], "big"), "?"),
        "sender_mac":  fmt_mac(p[8:14]),
        "sender_ip":   socket.inet_ntoa(p[14:18]),
        "target_mac":  fmt_mac(p[18:24]),
        "target_ip":   socket.inet_ntoa(p[24:28]),
    }


# ── Step 8: Pretty Print ──────────────────────────────────────────────────────

def print_frame(frame: dict, idx: int = 0):
    sep = "─" * 60
    print(f"\n{'═'*60}")
    print(f"  FRAME #{idx}")
    print(f"{'═'*60}")

    if "error" in frame:
        print(f"  ⚠  {frame['error']}")
        return

    print(f"  Dst MAC   : {frame.get('dst_mac','?')}")
    print(f"  Src MAC   : {frame.get('src_mac','?')}")
    print(f"  EtherType : {frame.get('ethertype','?')}  ({frame.get('proto_name','?')})")

    if "payload_len" in frame:
        print(f"  Payload   : {frame['payload_len']} bytes")
        fcs_ok = "✓ valid" if frame.get("fcs_valid") else "✗ MISMATCH"
        print(f"  FCS       : {frame.get('fcs_received','?')}  [{fcs_ok}]")

        hex_payload = frame.get("payload_hex", "")
        if hex_payload:
            print(f"\n  Payload hex (first 64 bytes):")
            for i in range(0, min(128, len(hex_payload)), 32):
                chunk = hex_payload[i:i+32]
                pairs = " ".join(chunk[j:j+2] for j in range(0, len(chunk), 2))
                print(f"    {pairs}")

    if "ipv4" in frame:
        v = frame["ipv4"]
        print(f"\n  ▶ IPv4  {v['src']} → {v['dst']}"
              f"  proto={v['proto']}  TTL={v['ttl']}")

    if "arp" in frame:
        a = frame["arp"]
        print(f"\n  ▶ ARP  {a['operation']}"
              f"  {a['sender_ip']} ({a['sender_mac']})"
              f" → {a['target_ip']} ({a['target_mac']})")

    print(sep)


# ── Main Pipeline ─────────────────────────────────────────────────────────────

def decode_capture(samples: np.ndarray, verbose: bool = True) -> list[dict]:
    """
    Full decode pipeline.

    Parameters
    ----------
    samples : np.ndarray  shape=(N,)  raw voltage samples at 1.25 GSa/s
    verbose : bool        print decoded frames to stdout

    Returns
    -------
    List of parsed frame dicts
    """
    print(f"[1/7] Input: {len(samples)} samples  "
          f"({len(samples)/SAMPLE_RATE*1e6:.1f} µs  @  "
          f"{SAMPLE_RATE/1e9:.2f} GSa/s)")

    # 1. Pre-process
    sig = remove_dc_offset(samples)
    sig = apply_lowpass(sig)

    # 2. Threshold
    thr = detect_threshold(sig)
    print(f"[2/7] Threshold: {thr:.4f} V")

    # 3. Edge detection
    edges = find_edges(sig, thr)
    print(f"[3/7] Edges found: {len(edges)}")

    if len(edges) < 8:
        print("      ⚠  Too few edges — check signal amplitude / polarity")
        return []

    # 4. Clock recovery
    bit_centres = recover_clock(edges, SAMPLES_PER_BIT)
    print(f"[4/7] Recovered bit centres: {len(bit_centres)}")

    # 5. Manchester decode
    raw_bits = decode_manchester(sig, bit_centres, thr)
    valid    = sum(1 for b in raw_bits if b >= 0)
    print(f"[5/7] Decoded bits: {len(raw_bits)}  ({valid} valid, "
          f"{len(raw_bits)-valid} invalid symbols)")

    # 6. Find all frames in the capture (multiple frames may be present)
    frames = []
    search_start = 0

    while search_start < len(raw_bits):
        offset = find_sfd(raw_bits[search_start:])
        if offset == -1:
            break

        abs_offset = search_start + offset
        # Grab up to 1518 bytes (max Ethernet frame) = 12144 bits
        frame_bits = raw_bits[abs_offset: abs_offset + 12144]
        frame_bits = [b for b in frame_bits if b >= 0]   # strip invalids

        if len(frame_bits) < 112:     # less than 14 bytes → skip
            search_start = abs_offset + 8
            continue

        frame_bytes = bits_to_bytes(frame_bits)
        parsed      = parse_ethernet_frame(frame_bytes)
        frames.append(parsed)

        # Advance past this frame
        search_start = abs_offset + len(frame_bits)

    print(f"[6/7] Frames decoded: {len(frames)}")

    # 7. Output
    if verbose:
        if frames:
            for i, f in enumerate(frames):
                print_frame(f, i)
        else:
            print("\n  No valid Ethernet frames found in capture.")
            print("  Tips:")
            print("   • Check signal polarity (try -samples)")
            print("   • Verify probe calibration / DC offset")
            print("   • Ensure capture includes the preamble")

    return frames


# ── Demo with synthetic signal ─────────────────────────────────────────────────

def _make_synthetic_capture() -> np.ndarray:
    """
    Generate a synthetic 10BASE-T frame for testing when no real capture
    is available.  Encodes a minimal ARP REQUEST frame with Manchester coding.
    """
    # Build frame bytes
    dst   = bytes([0xFF,0xFF,0xFF,0xFF,0xFF,0xFF])   # broadcast
    src   = bytes([0xDE,0xAD,0xBE,0xEF,0x00,0x01])
    etype = bytes([0x08,0x06])                        # ARP

    # Minimal ARP payload (28 bytes)
    import socket
    arp = (b'\x00\x01'                   # hw type: Ethernet
           b'\x08\x00'                   # proto: IPv4
           b'\x06\x04'                   # hw/proto addr lengths
           b'\x00\x01'                   # operation: REQUEST
           + src                         # sender MAC
           + socket.inet_aton('192.168.1.10')   # sender IP
           + bytes(6)                    # target MAC (unknown)
           + socket.inet_aton('192.168.1.1'))    # target IP

    payload = dst + src + etype + arp
    # Pad to 60 bytes
    payload += bytes(max(0, 60 - len(payload)))
    # Append CRC
    crc = crc32_ethernet(payload)
    frame_bytes = payload + crc.to_bytes(4, "little")

    # Encode as bits (MSB first per byte)
    data_bits = []
    for byte in frame_bytes:
        for bit in range(7, -1, -1):
            data_bits.append((byte >> bit) & 1)

    # Prepend preamble + SFD
    preamble = [1, 0] * 28
    sfd      = [1, 0, 1, 0, 1, 0, 1, 1]
    all_bits = preamble + sfd + data_bits

    # Manchester encode
    hi, lo   = 0.7, -0.7
    sig      = []
    for bit in all_bits:
        first_half  = lo if bit == 1 else hi
        second_half = hi if bit == 1 else lo
        sig += [first_half] * HALF_BIT + [second_half] * HALF_BIT

    # Add noise and idle gaps
    sig_arr = np.array(sig, dtype=np.float32)
    noise   = np.random.normal(0, 0.015, len(sig_arr))

    # Pad to 250 000 samples with idle (random noise around 0)
    total   = 250_000
    idle    = np.random.normal(0, 0.005, total - len(sig_arr))
    full    = np.concatenate([
        np.random.normal(0, 0.005, 500),   # quiet lead-in
        sig_arr + noise,
        idle
    ])[:total]

    return full.astype(np.float32)


# ── Entry point ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import sys

    # ── Option A: load your real capture ──────────────────────────────────────
    # samples = np.load("your_capture.npy")

    # ── Option B: paste / assign your array here ──────────────────────────────
    # samples = np.array([ 0.0080625, -0.0024375, ... ], dtype=np.float32)

    # ── Option C: run with synthetic demo data ────────────────────────────────
    print("=" * 60)
    print("  10BASE-T Ethernet Decoder")
    print("  Running on SYNTHETIC demo capture")
    print("=" * 60)
    np.random.seed(42)
    samples = _make_synthetic_capture()

    frames = decode_capture(packets[0,:], verbose=True)

    # Programmatic access to results:
    # for f in frames:
    #     print(f["dst_mac"], f["src_mac"], f["proto_name"])

  10BASE-T Ethernet Decoder
  Running on SYNTHETIC demo capture
[1/7] Input: 250000 samples  (200.0 µs  @  1.25 GSa/s)
[2/7] Threshold: -0.0005 V
[3/7] Edges found: 6115
[4/7] Recovered bit centres: 1894
[5/7] Decoded bits: 1894  (1062 valid, 832 invalid symbols)
[6/7] Frames decoded: 0

  No valid Ethernet frames found in capture.
  Tips:
   • Check signal polarity (try -samples)
   • Verify probe calibration / DC offset
   • Ensure capture includes the preamble
